In [8]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import chi2
import scipy.constants as sc
import pandas as pd

In [2]:
from opt_analysis_v import analyze_isotope_combined_vfit


# -----------------------------
# Coil / geometry inputs
# -----------------------------
mu0 = 4*np.pi*1e-7   # T m / A
N = 135
R = 27.5e-2          # m
h = 14.25e-2         # m
z = 0.0              # m

# -----------------------------
# Fit results from your analysis
# -----------------------------
res_85 = {
    "isotope": "Rb85",
    "I_nuclear": 5/2,
    "A_MHz_per_A": 2.0423,
    "A_err_MHz_per_A": 0.0172,
    "I0_A": -0.0916,
    "I0_err_A": 0.0014,
    "f0_MHz": -0.0175,
    "f0_err_MHz": 0.0206,
    "B_amb_G": 0.3953,
    "B_amb_err_G": 0.0063,
}

res_87 = {
    "isotope": "Rb87",
    "I_nuclear": 3/2,
    "A_MHz_per_A": 3.1104,
    "A_err_MHz_per_A": 0.0667,
    "I0_A": -0.0879,
    "I0_err_A": 0.0027,
    "f0_MHz": -0.0961,
    "f0_err_MHz": 0.0771,
    "B_amb_G": 0.3797,
    "B_amb_err_G": 0.0117,
}

In [3]:
def nu_over_B_MHz_per_G(I_nuclear):
    """
    Eq. (6): nu / B_exp = 2.799 / (2I+1) MHz/G
    """
    return 2.799 / (2*I_nuclear + 1)

def Bexp_from_nu_G(nu_MHz, I_nuclear):
    """
    Convert measured resonance frequency to B_exp in gauss.
    """
    return nu_MHz / nu_over_B_MHz_per_G(I_nuclear)

def Bexp_err_from_nu_err_G(nu_err_MHz, I_nuclear):
    """
    Propagate uncertainty from frequency to field.
    """
    return nu_err_MHz / nu_over_B_MHz_per_G(I_nuclear)

def helmholtz_pair_field_G(I_A, N, R, h, z=0.0):
    """
    Field at position z on the axis of a two-coil pair.
    Returns field in gauss.
    """
    term1 = 1.0 / (R**2 + (z - h/2)**2)**(3/2)
    term2 = 1.0 / (R**2 + (z + h/2)**2)**(3/2)
    B_T = (mu0 * N * R**2 / 2.0) * (term1 + term2) * I_A
    return B_T * 1e4  # T -> G

def helmholtz_pair_calibration_G_per_A(N, R, h, z=0.0):
    return helmholtz_pair_field_G(1.0, N, R, h, z)

Bcal = helmholtz_pair_calibration_G_per_A(N, R, h, z)
print(f"Coil calibration = {Bcal:.4f} G/A")

Coil calibration = 5.5961 G/A


In [4]:
# ambient field

def Bambient_from_I0_G(I0_A, I0_err_A, Bcal_G_per_A):
    B = abs(I0_A) * Bcal_G_per_A
    dB = Bcal_G_per_A * I0_err_A
    return B, dB

for res in [res_85, res_87]:
    Bgeom, dBgeom = Bambient_from_I0_G(res["I0_A"], res["I0_err_A"], Bcal)
    print(f'{res["isotope"]}:')
    print(f'  fit-reported B_amb = {res["B_amb_G"]:.4f} ± {res["B_amb_err_G"]:.4f} G')
    print(f'  geometry+I0 B_amb  = {Bgeom:.4f} ± {dBgeom:.4f} G')

Rb85:
  fit-reported B_amb = 0.3953 ± 0.0063 G
  geometry+I0 B_amb  = 0.5126 ± 0.0078 G
Rb87:
  fit-reported B_amb = 0.3797 ± 0.0117 G
  geometry+I0 B_amb  = 0.4919 ± 0.0151 G


In [5]:
def analyze_single_point(nu_MHz, nu_err_MHz, Icoil_A, Icoil_err_A, fit_res,
                         N, R, h, z=0.0, use_fit_ambient=True):
    """
    For one data point:
      1) compute B_exp from Eq. (6)
      2) compute B_coil from geometry and current
      3) compute total field at bulb by adding/subtracting ambient field

    Sign convention:
      We take B_total = |B_coil + B_amb|, where B_coil carries the sign of current.
      Since B_coil can be positive or negative, this naturally handles +I and -I.
    """
    I_nuclear = fit_res["I_nuclear"]

    # Experimental field from resonance frequency
    B_exp_G = Bexp_from_nu_G(nu_MHz, I_nuclear)
    dB_exp_G = Bexp_err_from_nu_err_G(nu_err_MHz, I_nuclear)

    # Coil-only field from geometry
    B_coil_G = helmholtz_pair_field_G(Icoil_A, N, R, h, z)
    dB_coil_G = abs(Bcal * Icoil_err_A)

    # Ambient field choice
    if use_fit_ambient:
        B_amb_G = fit_res["B_amb_G"]
        dB_amb_G = fit_res["B_amb_err_G"]
        amb_source = "fit-reported"
    else:
        B_amb_G, dB_amb_G = Bambient_from_I0_G(fit_res["I0_A"], fit_res["I0_err_A"], Bcal)
        amb_source = "geometry + I0"

    # Total field at bulb
    B_total_G = abs(B_coil_G + B_amb_G)
    dB_total_G = np.sqrt(dB_coil_G**2 + dB_amb_G**2)

    # Comparison
    diff_G = B_exp_G - B_total_G
    percent_diff = 100 * abs(diff_G) / B_total_G if B_total_G != 0 else np.nan

    return {
        "isotope": fit_res["isotope"],
        "nu_MHz": nu_MHz,
        "nu_err_MHz": nu_err_MHz,
        "Icoil_A": Icoil_A,
        "Icoil_err_A": Icoil_err_A,
        "B_exp_G": B_exp_G,
        "B_exp_err_G": dB_exp_G,
        "B_coil_G": B_coil_G,
        "B_coil_err_G": dB_coil_G,
        "B_amb_G": B_amb_G,
        "B_amb_err_G": dB_amb_G,
        "ambient_source": amb_source,
        "B_total_G": B_total_G,
        "B_total_err_G": dB_total_G,
        "difference_G": diff_G,
        "percent_difference": percent_diff,
    }

In [9]:
# calculation for one point

nu_plus = 3.0
nu_plus_err = 0.05
I_plus = 1.40464
I_plus_err = 0.01523

nu_minus = 3
nu_minus_err = 0.05
I_minus = -1.567505
I_minus_err = 0.004505

out_plus_85 = analyze_single_point(
    nu_MHz=nu_plus,
    nu_err_MHz=nu_plus_err,
    Icoil_A=I_plus,
    Icoil_err_A=I_plus_err,
    fit_res=res_85,
    N=N, R=R, h=h, z=z,
    use_fit_ambient=True
)

out_minus_85 = analyze_single_point(
    nu_MHz=nu_minus,
    nu_err_MHz=nu_minus_err,
    Icoil_A=I_minus,
    Icoil_err_A=I_minus_err,
    fit_res=res_85,
    N=N, R=R, h=h, z=z,
    use_fit_ambient=True
)

pd.DataFrame([out_plus_85, out_minus_85])

,isotope,nu_MHz,nu_err_MHz,Icoil_A,Icoil_err_A,B_exp_G,B_exp_err_G,B_coil_G,B_coil_err_G,B_amb_G,B_amb_err_G,ambient_source,B_total_G,B_total_err_G,difference_G,percent_difference
0,Rb85,3.0,0.05,1.404640,0.015230,6.430868,0.107181,7.860517,0.085229,0.3953,0.0063,fit-reported,8.255817,0.085461,-1.824949,22.105010
1,Rb85,3.0,0.05,-1.567505,0.004505,6.430868,0.107181,-8.771928,0.025210,0.3953,0.0063,fit-reported,8.376628,0.025986,-1.945759,23.228434


# Q5

In [11]:
def summarize_ambient(res, Bcal):
    Bgeom, dBgeom = Bambient_from_I0_G(res["I0_A"], res["I0_err_A"], Bcal)

    diff = res["B_amb_G"] - Bgeom
    sigma = np.sqrt(res["B_amb_err_G"]**2 + dBgeom**2)
    nsigma = diff / sigma
    percent_diff = 100 * abs(diff) / Bgeom

    return {
        "isotope": res["isotope"],
        "fit_Bamb_G": res["B_amb_G"],
        "fit_Bamb_err_G": res["B_amb_err_G"],
        "geom_Bamb_G": Bgeom,
        "geom_Bamb_err_G": dBgeom,
        "difference_G": diff,
        "combined_sigma_G": sigma,
        "difference_in_sigma": nsigma,
        "percent_difference": percent_diff,
    }

summary_df = pd.DataFrame([
    summarize_ambient(res_85, Bcal),
    summarize_ambient(res_87, Bcal)
])

summary_df

,isotope,fit_Bamb_G,fit_Bamb_err_G,geom_Bamb_G,geom_Bamb_err_G,difference_G,combined_sigma_G,difference_in_sigma,percent_difference
0,Rb85,0.3953,0.0063,0.512604,0.007835,-0.117304,0.010053,-11.668082,22.883868
1,Rb87,0.3797,0.0117,0.491898,0.015109,-0.112198,0.019110,-5.871205,22.809187


In [12]:
# weighted ave of ambient field

def weighted_average(values, errors):
    values = np.asarray(values, dtype=float)
    errors = np.asarray(errors, dtype=float)
    w = 1.0 / errors**2
    avg = np.sum(w * values) / np.sum(w)
    err = np.sqrt(1.0 / np.sum(w))
    return avg, err

Bavg, dBavg = weighted_average(
    [res_85["B_amb_G"], res_87["B_amb_G"]],
    [res_85["B_amb_err_G"], res_87["B_amb_err_G"]]
)

print(f"Weighted average ambient field = {Bavg:.4f} ± {dBavg:.4f} G")


Weighted average ambient field = 0.3918 ± 0.0055 G


In [14]:
# single point results

def print_single_point_result(out):
    print(f'=== {out["isotope"]} single-point comparison ===')
    print(f'nu              = {out["nu_MHz"]:.4f} ± {out["nu_err_MHz"]:.4f} MHz')
    print(f'I_coil          = {out["Icoil_A"]:.4f} ± {out["Icoil_err_A"]:.4f} A')
    print(f'B_exp           = {out["B_exp_G"]:.4f} ± {out["B_exp_err_G"]:.4f} G')
    print(f'B_coil          = {out["B_coil_G"]:.4f} ± {out["B_coil_err_G"]:.4f} G')
    print(f'B_amb ({out["ambient_source"]}) = {out["B_amb_G"]:.4f} ± {out["B_amb_err_G"]:.4f} G')
    print(f'B_total at bulb = {out["B_total_G"]:.4f} ± {out["B_total_err_G"]:.4f} G')
    print(f'difference      = {out["difference_G"]:.4f} G')
    print(f'percent diff    = {out["percent_difference"]:.2f} %')


print_single_point_result(out_plus_85)
print()
print_single_point_result(out_minus_85)

=== Rb85 single-point comparison ===
nu              = 3.0000 ± 0.0500 MHz
I_coil          = 1.4046 ± 0.0152 A
B_exp           = 6.4309 ± 0.1072 G
B_coil          = 7.8605 ± 0.0852 G
B_amb (fit-reported) = 0.3953 ± 0.0063 G
B_total at bulb = 8.2558 ± 0.0855 G
difference      = -1.8249 G
percent diff    = 22.11 %

=== Rb85 single-point comparison ===
nu              = 3.0000 ± 0.0500 MHz
I_coil          = -1.5675 ± 0.0045 A
B_exp           = 6.4309 ± 0.1072 G
B_coil          = -8.7719 ± 0.0252 G
B_amb (fit-reported) = 0.3953 ± 0.0063 G
B_total at bulb = 8.3766 ± 0.0260 G
difference      = -1.9458 G
percent diff    = 23.23 %
